In [12]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================================
# 0. DIRECTORY PATH FIX 
# =========================================================================
# This appends the parent folder (the root of the repo) to Python's path
# so it can successfully find and import the 'models' directory from inside 
# your 'research' folder.
sys.path.append(os.path.abspath('..')) 

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp
from models.frameworks import IsoAlign

In [18]:
import os
import glob
import numpy as np
import torch
import mne
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

data_dir = "/home/gella.saikrishna/sleep-edf-database-expanded-1.0.0/sleep-edf-database-expanded-1.0.0/sleep-cassette"
output_pt = "data/sleep_multichannel_3c.pt"

target_channels = ['EEG Fpz-Cz', 'EOG horizontal', 'EMG submental']
sfreq = 100

event_mapping = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3,
    'Sleep stage R': 4
}

tmax = 30.0 - (1.0 / sfreq)
all_data = []
all_labels = []

psg_files = sorted(glob.glob(os.path.join(data_dir, "*PSG.edf")))
hyp_files = sorted(glob.glob(os.path.join(data_dir, "*Hypnogram.edf")))

print(f"Found {len(psg_files)} PSG files and {len(hyp_files)} Hypnogram files.")
print("Starting extraction...")

for i, (psg_f, hyp_f) in enumerate(zip(psg_files, hyp_files)):
    try:
        raw = mne.io.read_raw_edf(psg_f, preload=True)
        available_channels = raw.ch_names
        
        valid_channels = [ch for ch in target_channels if ch in available_channels]
        if len(valid_channels) != 3:
            continue
            
        raw.pick_channels(valid_channels)
        raw.reorder_channels(target_channels)
        raw.resample(sfreq)
        
        annot = mne.read_annotations(hyp_f)
        raw.set_annotations(annot, emit_warning=False)
        
        events, _ = mne.events_from_annotations(
            raw, event_id=event_mapping, chunk_duration=30.0
        )
        
        epochs = mne.Epochs(raw, events, event_id=None, tmin=0, tmax=tmax, 
                            baseline=None, preload=True, on_missing='ignore')
        
        data = epochs.get_data()
        labels = epochs.events[:, 2]
        
        all_data.append(data)
        all_labels.append(labels)
        
    except Exception as e:
        print(f"Error processing {psg_f}: {e}")
        pass

    if (i + 1) % 5 == 0:
        print(f"[{i + 1}/{len(psg_files)}] files processed...")

print("Concatenating data...")
final_data = np.concatenate(all_data, axis=0)
final_labels = np.concatenate(all_labels, axis=0)

print(f"Total epochs extracted: {len(final_data)}")
print("Applying 64/16/20 Stratified Split...")

X_train, X_temp, y_train, y_temp = train_test_split(
    final_data, final_labels, test_size=0.36, random_state=42, stratify=final_labels
)

test_ratio = 20.0 / 36.0 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=test_ratio, random_state=42, stratify=y_temp
)

print(f"Split Complete:")
print(f"  Train: {len(X_train)} epochs (approx 64%)")
print(f"  Val:   {len(X_val)} epochs (approx 16%)")
print(f"  Test:  {len(X_test)} epochs (approx 20%)")

dataset_dict = {
    'train': {
        'samples': torch.FloatTensor(X_train),
        'labels': torch.LongTensor(y_train)
    },
    'val': {
        'samples': torch.FloatTensor(X_val),
        'labels': torch.LongTensor(y_val)
    },
    'test': {
        'samples': torch.FloatTensor(X_test),
        'labels': torch.LongTensor(y_test)
    }
}

os.makedirs(os.path.dirname(output_pt), exist_ok=True)
torch.save(dataset_dict, output_pt)
print(f"✅ Successfully saved split multi-channel dataset to {output_pt}")

Found 153 PSG files and 153 Hypnogram files.
Starting extraction...
[5/153] files processed...
[10/153] files processed...
[15/153] files processed...
[20/153] files processed...
[25/153] files processed...
[30/153] files processed...
[35/153] files processed...
[40/153] files processed...
[45/153] files processed...
[50/153] files processed...
[55/153] files processed...
[60/153] files processed...
[65/153] files processed...
[70/153] files processed...
[75/153] files processed...
[80/153] files processed...
[85/153] files processed...
[90/153] files processed...
[95/153] files processed...
[100/153] files processed...
[105/153] files processed...
[110/153] files processed...
[115/153] files processed...
[120/153] files processed...
[125/153] files processed...
[130/153] files processed...
[135/153] files processed...
[140/153] files processed...
[145/153] files processed...
[150/153] files processed...
Concatenating data...
Total epochs extracted: 414961
Applying 64/16/20 Stratified 

In [29]:
import torch

# Dataset path
dataset_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"

# Load dictionary
data_obj = torch.load(dataset_path, map_location="cpu")

print("--- Split Content Summary ---")
for split in ["train", "val", "test"]:
    if split in data_obj:
        split_dict = data_obj[split]
        
        # Extract samples and labels tensors
        samples = split_dict.get("samples", None)
        labels = split_dict.get("labels", None)
        
        print(f"\nSplit '{split}':")
        if samples is not None:
            print(f"  -> Total Epochs (Samples): {samples.shape[0]}")
            print(f"  -> Sample Dimensions:     {list(samples.shape)}")
        if labels is not None:
            # If it is a PyTorch tensor, find the unique classification categories
            if isinstance(labels, torch.Tensor):
                unique_classes = torch.unique(labels).tolist()
            else:
                unique_classes = list(set(labels))
            print(f"  -> Target Labels Found:   {len(labels)} values across {len(unique_classes)} unique sleep stages ({unique_classes})")
    else:
        print(f"Split '{split}' not found.")

--- Split Content Summary ---

Split 'train':
  -> Total Epochs (Samples): 265575
  -> Sample Dimensions:     [265575, 3, 3000]
  -> Target Labels Found:   265575 values across 5 unique sleep stages ([0, 1, 2, 3, 4])

Split 'val':
  -> Total Epochs (Samples): 66393
  -> Sample Dimensions:     [66393, 3, 3000]
  -> Target Labels Found:   66393 values across 5 unique sleep stages ([0, 1, 2, 3, 4])

Split 'test':
  -> Total Epochs (Samples): 82993
  -> Sample Dimensions:     [82993, 3, 3000]
  -> Target Labels Found:   82993 values across 5 unique sleep stages ([0, 1, 2, 3, 4])


In [ ]:
import torch
from torch import nn


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp

from models.frameworks import IsoAlign # Import the base class from your library

class MultiChannelIsoAlign(IsoAlign):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x1, x2, x3):
        # 1. DEBUG: Inspect shape entering the forward pass
        # print(f"DEBUG - x1 shape entering forward: {x1.shape}") 
        
        # 2. ResNet1D in models_nc.py expects (B, C, T) 
        # Inside ResNet1D.forward, it does x = x.transpose(-1,-2) 
        # which turns (B, C, T) into (B, T, C) for the conv layers.
        # Therefore, we pass x1 as is.
        
        _, R_t = self.encoder(x1)
        
        # 3. Handle spectrograms (x2)
        # Ensure it matches what UNET_2D_simp expects.
        # If UNET_2D_simp expects (B, C, F, T), do not permute if already in that shape.
        _, R_f = self.spect_encoder(x2)
        
        # 4. Handle Fourier
        _, R_f_FT = self.FT_encoder(x3)

        # 5. Project and Calculate Loss
        R_t = self.projector(R_t)
        R_f = self.projector_spect(R_f)
        R_f_FT = self.projector_FT(R_f_FT)

        return self.calc_loss(R_t, R_f, R_f_FT)
# =========================================================================
# 1. FIXED DATASET CLASS (Adapted for Multi-Channel Fusion)
# =========================================================================
class SleepEDF_HF_Dataset(Dataset):
    def __init__(self, pt_file_path="/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt", split="train"):
        """
        Loads the preprocessed Sleep-EDF dataset from a Hugging Face .pt file.
        split: 'train', 'test', or 'val'
        """
        print(f"Loading Hugging Face dataset from {pt_file_path}...")
        
        # 1. Load the PyTorch file directly to cpu memory first
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        # 2. Extract the sequences from nested dictionaries
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                print(f"Found '{split}' split, extracting...")
                data_obj = data_obj[split]
                
            if "samples" in data_obj:
                self.data = data_obj["samples"]
            elif "data" in data_obj:
                self.data = data_obj["data"]
            elif "x_data" in data_obj:
                self.data = data_obj["x_data"]
            elif "X_train" in data_obj:
                self.data = data_obj["X_train"]
            else:
                raise ValueError(f"Could not find the tensor. Available keys: {data_obj.keys()}")
        else:
            self.data = data_obj
            
        # 3. Ensure it is a FloatTensor
        if not isinstance(self.data, torch.Tensor):
            self.data = torch.FloatTensor(self.data)
        else:
            self.data = self.data.float()
            
        # 4. Handle Channel Dimension (Ensuring shape is [N, C, 3000])
        # If the data is [N, 3000], we expand it. If it's already [N, C, 3000], we keep it.
        if self.data.dim() == 2:
            self.data = self.data.unsqueeze(1)
            
        print(f"✅ Successfully loaded {len(self.data)} epochs from split '{split}'.")
        print(f"   Raw tensor shape: {self.data.shape}")
        
        # Pre-compute the Hann window for the Gabor/Wavelet transform
        self.window = torch.hann_window(128)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        # x_t shape is now (C, 3000)
        x_t = self.data[idx] 
        
        # --- 1. TIME DOMAIN ---
        # Shape: (C, 3000) - No unsqueeze needed since channels act as depth
        x_time = x_t 
        
        # --- 2. FOURIER DOMAIN ---
        # Apply FFT along the last dimension (time) for all channels simultaneously
        x_fft = torch.fft.rfft(x_t, dim=-1)
        magnitude = torch.abs(x_fft)
        phase = torch.angle(x_fft)
        # Concatenate magnitude and phase along the channel dimension -> Shape: (C*2, F)
        x_fourier = torch.cat([magnitude, phase], dim=0)
        
        # --- 3. WAVELET / GABOR DOMAIN ---
        # torch.stft natively treats the first dimension (C) as a batch, processing all channels!
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        
        # FIX 1: Slices 65 frequencies down to exactly 64 for all channels
        # Shape becomes (C, 64, TimeSteps)
        x_wavelet = torch.abs(x_stft)[:, :64, :] 
        
        # FIX 2: Pad time steps from 47 to 48 for all channels
        # Shape becomes (C, 64, 48)
        x_wavelet = F.pad(x_wavelet, (0, 1)) 
        
        return x_time, x_fourier, x_wavelet


# =========================================================================
# 2. SETUP HYPERPARAMETERS & DEVICE CONFIGURATION (Switched to GPU 1)
# =========================================================================
# 🔄 Target the second GPU card (index 1). Fallback to CPU if CUDA isn't available.
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu") 
print(f"\nUsing device: {DEVICE}")

# [MULTI-CHANNEL SETUP]
N_CHANNELS = 3       # Adjust based on your dataset (e.g., 3 for Sleep-EDF EEG/EOG/EMG)
TIME_CHANNELS = N_CHANNELS
FT_CHANNELS = N_CHANNELS * 2  # Magnitude + Phase for each channel

BATCH_SIZE = 64
EPOCHS = 40
TIME_STEPS = 3000    
LATENT_DIM = 128     

# Mock arguments class required by IsoAlign
class Args:
    wo_OB = False
    wo_OF = False
args = Args()

# =========================================================================
# 3. INITIALIZE DATA LOADERS
# =========================================================================
dataset_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"
train_dataset = SleepEDF_HF_Dataset(pt_file_path=dataset_path, split="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Derive exact shapes from the dataset output
_, _, sample_w = train_dataset[0]
SPECT_FREQ = sample_w.shape[1]   
SPECT_TIME = sample_w.shape[2]   

print(f"Spectrogram dimensions configured to: {SPECT_FREQ} Freqs x {SPECT_TIME} Time steps")

# =========================================================================
# 4. INSTANTIATE ARCHITECTURE COMPONENTS WITH LIVE MONKEY-PATCHING
# =========================================================================
# A. Time Encoder (1D ResNet)
# [MULTI-CHANNEL UPDATE]: in_channels dynamically tracks N_CHANNELS
time_encoder = ResNet1D(
    in_channels=TIME_CHANNELS, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=True, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

# B. Wavelet/Spectrogram Encoder (2D UNet Setup)
# [MULTI-CHANNEL UPDATE]: input_channels dynamically tracks N_CHANNELS
spect_encoder = UNET_2D_simp(
    input_channels=TIME_CHANNELS, output_channels=LATENT_DIM, layer_n=32, 
    spect_freq=SPECT_FREQ, spect_time=SPECT_TIME, kernel_size=3
)

# MONKEY PATCH 1: Inject missing layers for (64, 48) configuration
spect_encoder.fc = nn.Linear(6, 1)
spect_encoder.fc2 = nn.Linear(8, 1)
spect_encoder = spect_encoder.to(DEVICE)

# C. Fourier Encoder Wrapper
class FourierWrapper(nn.Module):
    def __init__(self, in_length):
        super().__init__()
        # [MULTI-CHANNEL UPDATE]: in_channels is FT_CHANNELS (N_CHANNELS * 2)
        self.enc = FourierEncoder(in_channels=FT_CHANNELS, in_length=in_length, out_channels=LATENT_DIM)
        
        # MONKEY PATCH 2: Inject missing layers for length 3000
        if in_length == 3000:
            self.enc.fc_abs = nn.Linear(376, 1)
            self.enc.fc_angle = nn.Linear(376, 1)
            
    def forward(self, x):
        return None, self.enc(x)

ft_encoder = FourierWrapper(in_length=TIME_STEPS).to(DEVICE)

# D. Master IsoAlign Multi-View Framework
model = IsoAlign(
    backbone=time_encoder, spect_encoder=spect_encoder, FT_encoder=ft_encoder, 
    DEVICE=DEVICE, dim=LATENT_DIM, batch_size=BATCH_SIZE, args=args
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# =========================================================================
# 5. EXECUTE THE TRAINING LOOP WITH INTEGRATED CHECKPOINTING
# =========================================================================
print("\n🚀 Starting Self-Supervised Multi-Channel Pre-training on GPU-1...")

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_loss = float('inf')
save_every_n_epochs = 5 

# --- CHECKPOINT RECOVERY ---
start_epoch = 0
# CHANGED: Checkpoint name updated to reflect the fusion architecture
resume_path = os.path.join(checkpoint_dir, "best_fusion_encoder.pth") 

if os.path.exists(resume_path):
    print(f"\n🔄 Found checkpoint at {resume_path}. Loading...")
    
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(checkpoint) 
    
    print("✅ Model weights loaded successfully.")
else:
    print("\n⚠️ No checkpoint found. Starting training from scratch.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w) in enumerate(train_loader):
        batch_t = batch_t.to(DEVICE)
        batch_f = batch_f.to(DEVICE)
        
        # Permute spectrogram layout from (B, C, F, T) to (B, F, T, C)
        # This aligns the multi-channel depth with the UNET's expected spatial input
        batch_w = batch_w.permute(0, 2, 3, 1).to(DEVICE) 
        
        optimizer.zero_grad()
        
        # Forward pass calculates all intra-domain losses and cross-domain mapping errors
        loss = model(batch_t, batch_w, batch_f)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Average Loss: {avg_loss:.4f}")
    
    # Checkpoint Metric Tracking
    if avg_loss < best_loss:
        best_loss = avg_loss
        # CHANGED: Save the best model as best_fusion_encoder.pth
        best_model_path = os.path.join(checkpoint_dir, "best_fusion_encoder.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"   🌟 New best loss achieved! Saved encoder to: {best_model_path}")
        
    if (epoch + 1) % save_every_n_epochs == 0:
        # CHANGED: Save periodic checkpoints as fusion_encoder_epoch_X.pth
        checkpoint_path = os.path.join(checkpoint_dir, f"fusion_encoder_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_path)
        print(f"   (i) Full recovery checkpoint saved to: {checkpoint_path}")

print("\n🎉 Pre-training Complete!")


Using device: cuda:1
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
Found 'train' split, extracting...
✅ Successfully loaded 265575 epochs from split 'train'.
   Raw tensor shape: torch.Size([265575, 3, 3000])
Spectrogram dimensions configured to: 64 Freqs x 48 Time steps

🚀 Starting Self-Supervised Multi-Channel Pre-training on GPU-1...

🔄 Found checkpoint at checkpoints/best_fusion_encoder.pth. Loading...
✅ Model weights loaded successfully.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Subset

# 1. SETUP REPRODUCIBILITY
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 2. UPDATED DATASET (Must return labels)
class SleepEDF_Downstream_Dataset(SleepEDF_HF_Dataset):
    def __init__(self, pt_file_path="/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt", split="train"):
        # 1. Run the base class __init__ to load the data
        super().__init__(pt_file_path=pt_file_path, split=split)
        
        # 2. Load the labels from the file
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        # Navigate the dictionary structure to find the labels
        # Adjust the key 'labels' below if your file uses a different key like 'y_data' or 'targets'
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
            
            if "labels" in data_obj:
                self.labels = data_obj["labels"]
            elif "y_data" in data_obj:
                self.labels = data_obj["y_data"]
            else:
                raise ValueError(f"Could not find labels in {data_obj.keys()}")
        else:
            raise ValueError("Dataset file must be a dictionary to contain both data and labels.")
            
        # Ensure labels are a tensor
        if not isinstance(self.labels, torch.Tensor):
            self.labels = torch.tensor(self.labels, dtype=torch.long)
            
        print(f"✅ Labels loaded. Shape: {self.labels.shape}")

    def __getitem__(self, idx):
        # Get data from base class
        x_time, x_fourier, x_wavelet = super().__getitem__(idx)
        # Access the labels loaded in __init__
        label = self.labels[idx]
        return x_time, x_fourier, x_wavelet, label

# 3. DEFINE CLASSIFICATION HEAD (Multi-View Fusion)
class MultiViewClassifier(nn.Module):
    def __init__(self, encoder_model, latent_dim, num_classes):
        super().__init__()
        self.encoder = encoder_model
        # Concatenate R_t, R_f, and R_f_FT (3 * LATENT_DIM)
        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x1, x2, x3):
        # Extract features using pre-trained encoders
        _, R_t = self.encoder.encoder(x1)
        _, R_f = self.encoder.spect_encoder(x2)
        _, R_f_FT = self.encoder.FT_encoder(x3)
        
        # Project them
        R_t = self.encoder.projector(R_t)
        R_f = self.encoder.projector_spect(R_f)
        R_f_FT = self.encoder.projector_FT(R_f_FT)
        
        # Combine representations
        combined = torch.cat([R_t, R_f, R_f_FT], dim=1)
        return self.classifier(combined)

# 4. PREPARE 10% SUBSET
full_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="train")
subset_size = int(0.1 * len(full_dataset))
indices = np.arange(len(full_dataset))
np.random.shuffle(indices)
train_subset = Subset(full_dataset, indices[:subset_size])
train_loader_10 = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)


Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
Found 'train' split, extracting...
✅ Successfully loaded 265575 epochs from split 'train'.
   Raw tensor shape: torch.Size([265575, 3, 3000])
✅ Labels loaded. Shape: torch.Size([265575])


In [40]:
# 5. INITIALIZE MODEL
model.load_state_dict(torch.load("checkpoints/best_fusion_encoder.pth", map_location=DEVICE))
# Freeze the encoders
for param in model.parameters():
    param.requires_grad = False

clf_model = MultiViewClassifier(model, LATENT_DIM, num_classes=5).to(DEVICE)
optimizer = optim.Adam(clf_model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 6. TRAINING LOOP
print("Starting downstream fine-tuning with 10% data...")
for epoch in range(20):
    clf_model.train()
    total_loss = 0
    
    for batch_t, batch_f, batch_w, labels in train_loader_10:
        
        # 1. TIME DOMAIN: Counteract the internal transpose
        # DataLoader gives [64, 3, 3000]. We swap to [64, 3000, 3].
        # Inside ResNet1D, it will transpose back to [64, 3, 3000] for Conv1d.
        batch_t = batch_t.to(DEVICE).transpose(1, 2)
        
        # 2. SPECTROGRAM: No permutation needed! 
        # DataLoader already provides [Batch, Channels, Freq, Time] -> [64, 3, 64, 48]
        # This exactly matches UNET_2D_simp expectations.
        batch_w = batch_w.to(DEVICE) 
        
        # 3. FOURIER DOMAIN & LABELS
        batch_f = batch_f.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # --- OPTIMIZATION ---
        optimizer.zero_grad()
        
        outputs = clf_model(batch_t, batch_w, batch_f)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader_10)
    print(f"Epoch [{epoch+1}/20] | Loss: {avg_loss:.4f}")

print("Downstream training completed successfully!")

Starting downstream fine-tuning with 10% data...
Epoch [1/20] | Loss: 0.6427
Epoch [2/20] | Loss: 0.4826
Epoch [3/20] | Loss: 0.4667
Epoch [4/20] | Loss: 0.4577
Epoch [5/20] | Loss: 0.4546
Epoch [6/20] | Loss: 0.4489
Epoch [7/20] | Loss: 0.4438
Epoch [8/20] | Loss: 0.4424
Epoch [9/20] | Loss: 0.4356
Epoch [10/20] | Loss: 0.4335
Epoch [11/20] | Loss: 0.4345
Epoch [12/20] | Loss: 0.4327
Epoch [13/20] | Loss: 0.4286
Epoch [14/20] | Loss: 0.4279
Epoch [15/20] | Loss: 0.4255
Epoch [16/20] | Loss: 0.4237
Epoch [17/20] | Loss: 0.4213
Epoch [18/20] | Loss: 0.4184
Epoch [19/20] | Loss: 0.4175
Epoch [20/20] | Loss: 0.4185
Downstream training completed successfully!


In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

# 1. LOAD THE TEST DATASET
print("Loading Test Dataset...")
test_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="test")

# Note: shuffle=False is standard for evaluation so predictions align with original order
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 2. EVALUATION LOOP
def evaluate_on_test(model, loader, device):
    model.eval() # Switch to evaluation mode (disables dropout, fixes batchnorm)
    
    all_preds = []
    all_targets = []
    
    print(f"Evaluating {len(loader.dataset)} samples...")
    
    # Disable gradient calculation for faster inference and lower memory usage
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in loader:
            
            # --- 1. PRE-PROCESS TENSORS (Exact same as training loop) ---
            # DataLoader gives [B, 3, 3000]. Swap to [B, 3000, 3] to counteract internal ResNet transpose.
            batch_t = batch_t.to(device).transpose(1, 2)
            
            # Spectrogram is already [B, Channels, Freq, Time], no permute needed
            batch_w = batch_w.to(device)
            batch_f = batch_f.to(device)
            
            # --- 2. FORWARD PASS ---
            outputs = model(batch_t, batch_w, batch_f)
            
            # --- 3. GET PREDICTIONS ---
            # Get the index of the max log-probability (the predicted class)
            _, predicted = torch.max(outputs, 1)
            
            # Store predictions and true labels on CPU
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    return np.array(all_targets), np.array(all_preds)

# Run the evaluation
y_true, y_pred = evaluate_on_test(clf_model, test_loader, DEVICE)

# 3. PRINT STATISTICS
# Map indices to the actual sleep stage names for readability
target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']

print("\n" + "="*50)
print("📊 PER-CLASS METRICS (TEST SET)")
print("="*50)

# classification_report automatically calculates Precision, Recall, F1, and Accuracy
report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
print(report)

print("\n" + "="*50)
print("🧮 CONFUSION MATRIX")
print("="*50)
conf_matrix = confusion_matrix(y_true, y_pred)
print(conf_matrix)

Loading Test Dataset...
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
Found 'test' split, extracting...
✅ Successfully loaded 82993 epochs from split 'test'.
   Raw tensor shape: torch.Size([82993, 3, 3000])
✅ Labels loaded. Shape: torch.Size([82993])
Evaluating 82993 samples...

📊 PER-CLASS METRICS (TEST SET)
              precision    recall  f1-score   support

    Wake (0)     0.9089    0.9537    0.9308     57087
      N1 (1)     0.4063    0.1440    0.2127      4305
      N2 (2)     0.6870    0.8240    0.7493     13826
      N3 (3)     0.7726    0.5928    0.6709      2608
     REM (4)     0.6107    0.3524    0.4469      5167

    accuracy                         0.8413     82993
   macro avg     0.6771    0.5734    0.6021     82993
weighted avg     0.8230    0.8413    0.8250     82993


🧮 CONFUSION MATRIX
[[54446   378  1742   128   393]
 [ 1577   620  1675    10   423]
 [ 1501   273 11393   31

: 